Ячейка 1: Инициализация и загрузка данных

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, avg, round, desc

spark = SparkSession.builder \
    .appName("Lab2_MongoDB_Reports") \
    .config("spark.driver.memory", "2g") \
    .config("spark.jars", "/home/jovyan/jars/postgresql-42.7.1.jar") \
    .getOrCreate()

print(f"Spark: {spark.version}")

pg_url = "jdbc:postgresql://postgres:5432/bigdata_lab"
pg_props = {"user": "student", "password": "student123", "driver": "org.postgresql.Driver"}

df_date = spark.read.jdbc(url=pg_url, table="dim_date", properties=pg_props)
df_customer = spark.read.jdbc(url=pg_url, table="dim_customer", properties=pg_props)
df_seller = spark.read.jdbc(url=pg_url, table="dim_seller", properties=pg_props)
df_product = spark.read.jdbc(url=pg_url, table="dim_product", properties=pg_props)
df_store = spark.read.jdbc(url=pg_url, table="dim_store", properties=pg_props)
df_supplier = spark.read.jdbc(url=pg_url, table="dim_supplier", properties=pg_props)
df_fact = spark.read.jdbc(url=pg_url, table="fact_sale", properties=pg_props)

print("Таблицы звезды загружены из PostgreSQL")

Spark: 3.5.0
Таблицы звезды загружены из PostgreSQL


Ячейка 2: Функция записи в MongoDB

In [2]:
import json
import pymongo

def write_to_mongodb(df, collection_name):
    client = pymongo.MongoClient("mongodb://mongodb:27017/")
    db = client["reports"]
    collection = db[collection_name]
    
    collection.drop()
    
    rows = df.toJSON().collect()
    docs = [json.loads(row) for row in rows]
    
    if docs:
        collection.insert_many(docs)
        print(f"{collection_name}: {len(docs)} документов записано")
    else:
        print(f"{collection_name}: пустой DataFrame")
    
    client.close()

Ячейка 3: Витрина 1 — product_sales

In [3]:
product_mart = df_fact.join(df_product, "product_key", "left") \
    .groupBy("product_id", "product_name", "product_category") \
    .agg(
        sum("sale_quantity").alias("total_quantity"),
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        round(avg("product_rating"), 2).alias("avg_rating"),
        avg("product_reviews").cast("int").alias("total_reviews")
    )

print("Топ-5 продуктов:")
product_mart.orderBy(desc("total_revenue")).select("product_name", "total_revenue").show(5, truncate=False)

write_to_mongodb(product_mart, "product_sales")

Топ-5 продуктов:
+------------+-------------+
|product_name|total_revenue|
+------------+-------------+
|Bird Cage   |4005.98      |
|Cat Toy     |3784.44      |
|Bird Cage   |3751.09      |
|Bird Cage   |3682.52      |
|Bird Cage   |3645.94      |
+------------+-------------+
only showing top 5 rows

product_sales: 1000 документов записано


Ячейка 4: Витрина 2 — customer_sales

In [4]:
customer_mart = df_fact.join(df_customer, "customer_key", "left") \
    .groupBy("customer_id", "first_name", "last_name", "customer_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_spent"),
        count("sale_id").alias("purchase_count"),
        round(avg("sale_total_price"), 2).alias("avg_check")
    ) \
    .withColumnRenamed("customer_country", "country")

print("Топ-5 клиентов:")
customer_mart.orderBy(desc("total_spent")).select("first_name", "last_name", "total_spent").show(5, truncate=False)

write_to_mongodb(customer_mart, "customer_sales")

Топ-5 клиентов:
+----------+----------+-----------+
|first_name|last_name |total_spent|
+----------+----------+-----------+
|Hannie    |Braddon   |4005.98    |
|Mercy     |Antonomoli|3784.44    |
|Genni     |Schultze  |3751.09    |
|Herschel  |Chaff     |3682.52    |
|Ramsay    |Karran    |3645.94    |
+----------+----------+-----------+
only showing top 5 rows

customer_sales: 1000 документов записано


Ячейка 5: Витрина 3 — time_sales

In [5]:
time_mart = df_fact.join(df_date, "date_key", "left") \
    .groupBy("year", "month", "month_name", "quarter") \
    .agg(
        round(sum("sale_total_price"), 2).alias("monthly_revenue"),
        count("sale_id").alias("order_count"),
        round(avg("sale_total_price"), 2).alias("avg_order_size")
    ) \
    .orderBy("year", "month")

print("Продажи по месяцам:")
time_mart.select("year", "month_name", "monthly_revenue", "order_count").show(12, truncate=False)

write_to_mongodb(time_mart, "time_sales")

Продажи по месяцам:
+----+----------+---------------+-----------+
|year|month_name|monthly_revenue|order_count|
+----+----------+---------------+-----------+
|2021|January   |224158.54      |874        |
|2021|February  |192348.31      |739        |
|2021|March     |207282.2       |843        |
|2021|April     |206592.82      |837        |
|2021|May       |211764.86      |828        |
|2021|June      |215042.8       |822        |
|2021|July      |220496.51      |858        |
|2021|August    |221275.78      |897        |
|2021|September |210623.43      |839        |
|2021|October   |228743.32      |892        |
|2021|November  |200154.69      |801        |
|2021|December  |191368.86      |770        |
+----+----------+---------------+-----------+

time_sales: 12 документов записано


Ячейка 6: Витрина 4 — store_sales


In [6]:
store_mart = df_fact.join(df_store, "store_key", "left") \
    .groupBy("store_name", "store_city", "store_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        count("sale_id").alias("order_count"),
        round(avg("sale_total_price"), 2).alias("avg_check")
    )

print("Топ-5 магазинов:")
store_mart.orderBy(desc("total_revenue")).select("store_name", "store_city", "total_revenue").show(5, truncate=False)

write_to_mongodb(store_mart, "store_sales")

Топ-5 магазинов:
+----------+----------+-------------+
|store_name|store_city|total_revenue|
+----------+----------+-------------+
|Mynte     |Brunflo   |15751.71     |
|Quatz     |Gemuruh   |15176.64     |
|Jayo      |Colima    |13976.01     |
|Quinu     |Bélabo    |13952.88     |
|Realcube  |Cincinnati|13700.77     |
+----------+----------+-------------+
only showing top 5 rows

store_sales: 383 документов записано


Ячейка 7: Витрина 5 — supplier_sales

In [7]:
supplier_mart = df_fact.join(df_product.select("product_key", "supplier_key"), "product_key", "left") \
    .join(df_supplier, "supplier_key", "left") \
    .groupBy("supplier_name", "supplier_country") \
    .agg(
        round(sum("sale_total_price"), 2).alias("total_revenue"),
        count("sale_id").alias("sales_count")
    )

print("Топ-5 поставщиков:")
supplier_mart.orderBy(desc("total_revenue")).show(5, truncate=False)

write_to_mongodb(supplier_mart, "supplier_sales")

Топ-5 поставщиков:
+-------------+----------------+-------------+-----------+
|supplier_name|supplier_country|total_revenue|sales_count|
+-------------+----------------+-------------+-----------+
|NULL         |NULL            |2412919.15   |9550       |
|Flashdog     |Russia          |19089.69     |70         |
|Ozu          |Philippines     |13865.79     |50         |
|Divanoodle   |China           |13028.36     |50         |
|Brainbox     |Israel          |11528.62     |40         |
+-------------+----------------+-------------+-----------+
only showing top 5 rows

supplier_sales: 17 документов записано


Ячейка 8: Витрина 6 — product_quality

In [8]:
quality_mart = df_fact.join(df_product, "product_key", "left") \
    .groupBy("product_id", "product_name", "product_category", "product_rating", "product_reviews") \
    .agg(
        sum("sale_quantity").alias("total_sold"),
        round(sum("sale_total_price"), 2).alias("total_revenue")
    )

print("Топ-5 по рейтингу:")
quality_mart.orderBy(desc("product_rating")).select("product_name", "product_rating", "total_sold").show(5, truncate=False)

write_to_mongodb(quality_mart, "product_quality")

Топ-5 по рейтингу:
+------------+--------------+----------+
|product_name|product_rating|total_sold|
+------------+--------------+----------+
|Cat Toy     |5.0           |51        |
|Dog Food    |5.0           |60        |
|Dog Food    |5.0           |53        |
|Cat Toy     |5.0           |44        |
|Bird Cage   |5.0           |60        |
+------------+--------------+----------+
only showing top 5 rows

product_quality: 1000 документов записано


Ячейка 9: Проверка

In [9]:
client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["reports"]

collections = ["product_sales", "customer_sales", "time_sales", 
               "store_sales", "supplier_sales", "product_quality"]

for coll_name in collections:
    count = db[coll_name].count_documents({})
    print(f"{coll_name}: {count} документов")

client.close()
print("\nВсе витрины в MongoDB готовы")

product_sales: 1000 документов
customer_sales: 1000 документов
time_sales: 12 документов
store_sales: 383 документов
supplier_sales: 17 документов
product_quality: 1000 документов

Все витрины в MongoDB готовы


Ячейка 10: Пример содержимого

In [10]:
client = pymongo.MongoClient("mongodb://mongodb:27017/")
db = client["reports"]

for coll_name in ["product_sales", "customer_sales", "time_sales"]:
    print(f"\n=== {coll_name} ===")
    for doc in db[coll_name].find().limit(2):
        print(doc)

client.close()


=== product_sales ===
{'_id': ObjectId('6a0100a366dfd9cdc18da556'), 'product_id': 305, 'product_name': 'Cat Toy', 'product_category': 'Cage', 'total_quantity': 54, 'total_revenue': 2667.64, 'avg_rating': 4.1, 'total_reviews': 959}
{'_id': ObjectId('6a0100a366dfd9cdc18da557'), 'product_id': 25, 'product_name': 'Cat Toy', 'product_category': 'Cage', 'total_quantity': 51, 'total_revenue': 2326.48, 'avg_rating': 5.0, 'total_reviews': 882}

=== customer_sales ===
{'_id': ObjectId('6a0100a666dfd9cdc18da93f'), 'customer_id': 34, 'first_name': 'Colleen', 'last_name': 'Symmers', 'country': 'Brazil', 'total_spent': 2855.42, 'purchase_count': 10, 'avg_check': 285.54}
{'_id': ObjectId('6a0100a666dfd9cdc18da940'), 'customer_id': 246, 'first_name': 'Solomon', 'last_name': 'Haythornthwaite', 'country': 'Albania', 'total_spent': 2447.52, 'purchase_count': 10, 'avg_check': 244.75}

=== time_sales ===
{'_id': ObjectId('6a0100a766dfd9cdc18dad28'), 'year': 2021, 'month': 1, 'month_name': 'January', 'quar